# FIFA World Cup 2026 — Notebook 2: Modelling

| Model | Target | Method |
|---|---|---|
| Poisson match model | Home/away goals | Team attack & defence ratings |
| LOO cross-validation | Match prediction | Leave-one-tournament-out (1998–2022) |
| Tournament simulator | Winner probability | Monte Carlo 10,000 runs |
| LightGBM Ranker | Golden Boot | Pairwise ranking on player features |
| LightGBM Ranker | Best Player | Pairwise ranking |
| LightGBM Ranker | Young Player (U23) | Pairwise ranking, age-filtered |

**Inputs:** `team_features.csv`, `player_features.csv` (from Notebook 1)

## 1. Setup

In [ ]:
!pip install requests pandas numpy matplotlib seaborn scikit-learn scipy lightgbm --quiet

import requests, io, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import poisson
from sklearn.metrics import log_loss, brier_score_loss
from sklearn.calibration import calibration_curve
from collections import defaultdict
import lightgbm as lgb

warnings.filterwarnings("ignore")
np.random.seed(42)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid")
print("Ready")

## 2. Load Data

In [ ]:
try:
    team_features   = pd.read_csv("team_features.csv")
    player_features = pd.read_csv("player_features.csv")
    print(f"Loaded from CSV")
except FileNotFoundError:
    raise FileNotFoundError("Run Notebook 1 first to generate team_features.csv and player_features.csv")

# Fetch raw WC matches for the Poisson model (needs match-level data)
BASE = "https://raw.githubusercontent.com/jfjelstul/worldcup/master/data-csv/"
def fetch(f):
    r = requests.get(BASE + f, timeout=15); r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text), low_memory=False)

tournaments = fetch("tournaments.csv")
raw_matches  = fetch("matches.csv")

men_tourns = tournaments[tournaments["tournament_name"].str.contains("Men", na=False)]
YEAR_MAP   = men_tourns.set_index("tournament_id")["year"].to_dict()
WINNER_MAP = men_tourns.set_index("tournament_id")["winner"].to_dict()

def men_only(df):
    df = df.copy(); df["year"] = df["tournament_id"].map(YEAR_MAP)
    return df[df["year"].notna()].reset_index(drop=True)

matches = men_only(raw_matches)
matches["year"]            = matches["year"].astype(int)
matches["home_team_score"] = pd.to_numeric(matches["home_team_score"], errors="coerce")
matches["away_team_score"] = pd.to_numeric(matches["away_team_score"], errors="coerce")
matches["match_date"]      = pd.to_datetime(matches["match_date"], errors="coerce")
matches = matches.dropna(subset=["home_team_score","away_team_score"]).reset_index(drop=True)

team_features["year"]   = team_features["year"].astype(int)
player_features["year"] = player_features["year"].astype(int)

print(f"team_features   : {team_features.shape}")
print(f"player_features : {player_features.shape}")
print(f"matches         : {matches.shape}")
print(f"Years covered   : {sorted(matches['year'].unique())}")

## 3. Poisson Match Model

Goals modelled as:
```
home_goals ~ Poisson(attack[home] × defence[away] × home_advantage)
away_goals ~ Poisson(attack[away] × defence[home])
```
Ratings estimated iteratively. Higher attack = scores more. Lower defence = concedes less.

In [ ]:
def fit_poisson_ratings(df, n_iter=10):
    teams   = sorted(set(df["home_team_name"]) | set(df["away_team_name"]))
    attack  = {t: 1.0 for t in teams}
    defense = {t: 1.0 for t in teams}
    home_adv = df["home_team_score"].mean() / max(df["away_team_score"].mean(), 0.01)

    for _ in range(n_iter):
        for team in teams:
            h = df[df["home_team_name"] == team]
            a = df[df["away_team_name"] == team]

            actual_gf = list(h["home_team_score"]) + list(a["away_team_score"])
            actual_ga = list(h["away_team_score"]) + list(a["home_team_score"])

            exp_gf = (
                [attack[team] * defense.get(opp,1.0) * home_adv for opp in h["away_team_name"]] +
                [attack[team] * defense.get(opp,1.0)             for opp in a["home_team_name"]]
            )
            exp_ga = (
                [attack.get(opp,1.0) * defense[team]             for opp in h["away_team_name"]] +
                [attack.get(opp,1.0) * defense[team] * home_adv  for opp in a["home_team_name"]]
            )

            if sum(exp_gf) > 0: attack[team]  = sum(actual_gf) / sum(exp_gf)
            if sum(exp_ga) > 0: defense[team] = sum(actual_ga) / sum(exp_ga)

    return attack, defense, home_adv


AVG_ATTACK_GLOBAL  = 1.0
AVG_DEFENSE_GLOBAL = 1.0

def predict_match(home, away, attack, defense, home_adv,
                  avg_atk=None, avg_def=None):
    avg_atk = avg_atk or AVG_ATTACK_GLOBAL
    avg_def = avg_def or AVG_DEFENSE_GLOBAL
    lam_h = attack.get(home, avg_atk) * defense.get(away, avg_def) * home_adv
    lam_a = attack.get(away, avg_atk) * defense.get(home, avg_def)

    max_goals = 10
    p_home_win = p_draw = p_away_win = 0.0

    for hg in range(max_goals + 1):
        for ag in range(max_goals + 1):
            p = poisson.pmf(hg, lam_h) * poisson.pmf(ag, lam_a)
            if hg > ag:   p_home_win += p
            elif hg == ag: p_draw    += p
            else:          p_away_win += p

    return p_home_win, p_draw, p_away_win, lam_h, lam_a

print("Poisson functions defined.")

## 4. Leave-One-Tournament-Out (LOO) Cross-Validation

**Why LOO instead of a single 2018+2022 hold-out:**
Two tournaments can't distinguish a well-specified model from one that got lucky.
LOO across 1998–2022 (six tournaments) reports mean ± std of metrics, which shows
whether 2022's performance was representative or an outlier.

**What we measure per fold:**
- Result accuracy (home win / draw / away win)
- Brier score — lower is better (naive baseline ≈ 0.333)
- Log loss — lower is better
- Mean absolute error on expected goals

In [ ]:
LOO_YEARS = [y for y in sorted(matches["year"].unique()) if y >= 1998]

loo_results = []

for test_yr in LOO_YEARS:
    train_df = matches[matches["year"] < test_yr]
    test_df  = matches[matches["year"] == test_yr]

    if len(train_df) == 0 or len(test_df) == 0:
        continue

    atk, dfn, hadv = fit_poisson_ratings(train_df)
    avg_atk = np.mean(list(atk.values()))
    avg_def = np.mean(list(dfn.values()))

    correct = 0
    brier_scores = []
    log_losses   = []
    goal_maes    = []

    for _, row in test_df.iterrows():
        home, away = row["home_team_name"], row["away_team_name"]
        actual_hg, actual_ag = row["home_team_score"], row["away_team_score"]

        ph, pd_, pa, lh, la = predict_match(home, away, atk, dfn, hadv, avg_atk, avg_def)

        # Result accuracy
        pred_result   = ["H","D","A"][[ph,pd_,pa].index(max(ph,pd_,pa))]
        actual_result = "H" if actual_hg > actual_ag else ("D" if actual_hg == actual_ag else "A")
        correct += (pred_result == actual_result)

        # Brier score (binary: did home team win?)
        brier_scores.append((int(actual_hg > actual_ag) - ph)**2)

        # Log loss (3-class: H/D/A)
        probs  = np.clip([ph, pd_, pa], 1e-7, 1-1e-7)
        label  = ["H","D","A"].index(actual_result)
        y_true = [0,0,0]; y_true[label] = 1
        log_losses.append(-np.log(probs[label]))

        # Goal MAE
        goal_maes.append(abs(actual_hg - lh) + abs(actual_ag - la))

    n = len(test_df)
    loo_results.append({
        "year":     test_yr,
        "n_matches": n,
        "accuracy": correct / n,
        "brier":    np.mean(brier_scores),
        "log_loss": np.mean(log_losses),
        "goal_mae": np.mean(goal_maes),
    })

loo_df = pd.DataFrame(loo_results)
print("=== LOO Cross-Validation Results (1998–2022) ===")
display(loo_df.round(3))

print(f"\nMean accuracy : {loo_df['accuracy'].mean():.3f}  ± {loo_df['accuracy'].std():.3f}")
print(f"Mean Brier    : {loo_df['brier'].mean():.3f}  ± {loo_df['brier'].std():.3f}  (naive = 0.333)")
print(f"Mean log loss : {loo_df['log_loss'].mean():.3f}  ± {loo_df['log_loss'].std():.3f}")
print(f"Mean goal MAE : {loo_df['goal_mae'].mean():.3f}  ± {loo_df['goal_mae'].std():.3f}")
print("\nSkill score (Brier) =", round(1 - loo_df['brier'].mean() / 0.333, 3),
      " — fraction of naive baseline error eliminated")

## 5. Calibration Analysis

Does a 70% win probability really happen 70% of the time?
A well-calibrated model's reliability curve lies on the diagonal.

In [ ]:
# Collect all LOO predictions for calibration plot
all_ph, all_y = [], []

for test_yr in LOO_YEARS:
    train_df = matches[matches["year"] < test_yr]
    test_df  = matches[matches["year"] == test_yr]
    if len(train_df) == 0: continue

    atk, dfn, hadv = fit_poisson_ratings(train_df)
    avg_atk = np.mean(list(atk.values()))
    avg_def = np.mean(list(dfn.values()))

    for _, row in test_df.iterrows():
        home, away = row["home_team_name"], row["away_team_name"]
        ph, _, _, _, _ = predict_match(home, away, atk, dfn, hadv, avg_atk, avg_def)
        actual_win = int(row["home_team_score"] > row["away_team_score"])
        all_ph.append(ph)
        all_y.append(actual_win)

all_ph = np.array(all_ph)
all_y  = np.array(all_y)

frac_pos, mean_pred = calibration_curve(all_y, all_ph, n_bins=10)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(mean_pred, frac_pos, "o-", label="Poisson model", color="#E74C3C")
ax.plot([0,1],[0,1], "k--", alpha=0.5, label="Perfect calibration")
ax.set_xlabel("Predicted P(home win)")
ax.set_ylabel("Actual fraction of home wins")
ax.set_title("Calibration Plot — Home Win Probability (LOO 1998–2022)")
ax.legend()
plt.tight_layout()
plt.show()

overall_brier = np.mean((all_ph - all_y)**2)
print(f"Overall Brier score : {overall_brier:.3f}")
print(f"Overall log loss    : {log_loss(all_y, np.clip(all_ph, 1e-7, 1-1e-7)):.3f}")
print("\nIf the calibration curve bows above the diagonal, the model is over-confident.")
print("If it bows below, the model is under-confident.")

## 6. Train Final Poisson Model on All Historical Data

With validation done, we now train on all available data to maximise rating quality for 2026 simulation.

In [ ]:
attack, defense, home_adv = fit_poisson_ratings(matches)
AVG_ATTACK_GLOBAL  = np.mean(list(attack.values()))
AVG_DEFENSE_GLOBAL = np.mean(list(defense.values()))

print(f"Home advantage multiplier: {home_adv:.3f}")
print(f"\nTop 10 attack ratings:")
for team, val in sorted(attack.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {team:<28} {val:.3f}")
print(f"\nTop 10 defence ratings (lower = better defender):")
for team, val in sorted(defense.items(), key=lambda x: x[1])[:10]:
    print(f"  {team:<28} {val:.3f}")

## 7. Monte Carlo Tournament Simulator

10,000 simulations of the 2026 bracket.

**Group stage:** round-robin, top 2 advance on points → goal difference → goals scored.
**Knockout:** draws resolved by simulated penalty shootout (50/50).

**2026 format:** 48 teams, 12 groups of 4. Top 2 from each group (24 teams) + 8 best 3rd-placed teams advance to Round of 32.

In [ ]:
GROUPS_2026 = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Côte d'Ivoire", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cabo Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "Democratic Republic of the Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

# 2026: top 2 from each group (24) + best 8 third-placed teams = 32 advancing
def simulate_group_stage_2026(groups, attack, defense, avg_atk, avg_def):
    group_tables = {}
    third_placed = []

    for group_name, teams in groups.items():
        pts = {t: 0 for t in teams}
        gd  = {t: 0 for t in teams}
        gf  = {t: 0 for t in teams}

        for i, home in enumerate(teams):
            for away in teams[i+1:]:
                lh = attack.get(home, avg_atk) * defense.get(away, avg_def)
                la = attack.get(away, avg_atk) * defense.get(home, avg_def)
                hg = np.random.poisson(lh)
                ag = np.random.poisson(la)

                gd[home] += hg - ag; gd[away] += ag - hg
                gf[home] += hg;      gf[away] += ag

                if hg > ag:    pts[home] += 3
                elif hg == ag: pts[home] += 1; pts[away] += 1
                else:          pts[away] += 3

        ranked = sorted(teams, key=lambda t: (pts[t], gd[t], gf[t]), reverse=True)
        group_tables[group_name] = {
            "first": ranked[0], "second": ranked[1], "third": ranked[2],
            "third_pts": pts[ranked[2]], "third_gd": gd[ranked[2]], "third_gf": gf[ranked[2]],
        }

    # Pick best 8 third-placed teams
    thirds = sorted(
        group_tables.items(),
        key=lambda x: (x[1]["third_pts"], x[1]["third_gd"], x[1]["third_gf"]),
        reverse=True
    )[:8]
    third_qualifiers = [v["third"] for _, v in thirds]

    qualifiers = {g: (v["first"], v["second"]) for g, v in group_tables.items()}
    return qualifiers, third_qualifiers


def simulate_knockout_match(team_a, team_b, attack, defense, avg_atk, avg_def):
    lh = attack.get(team_a, avg_atk) * defense.get(team_b, avg_def)
    la = attack.get(team_b, avg_atk) * defense.get(team_a, avg_def)
    hg = np.random.poisson(lh)
    ag = np.random.poisson(la)
    if hg > ag: return team_a
    if ag > hg: return team_b
    return team_a if np.random.random() > 0.5 else team_b


def simulate_tournament_2026(groups, attack, defense, avg_atk, avg_def):
    qualifiers, thirds = simulate_group_stage_2026(groups, attack, defense, avg_atk, avg_def)

    # Build Round of 32: 24 group qualifiers + 8 best thirds
    # Seeded bracket: 1st-placed teams on one side, 2nd + 3rd on the other
    # Simplified: pair group winners vs runners-up in seeded order, thirds fill remaining slots
    group_order = list(groups.keys())
    r32 = []
    for g in group_order:
        r32.append(qualifiers[g][0])   # group winner
        r32.append(qualifiers[g][1])   # runner-up
    r32 += thirds  # 8 best thirds

    # If not 32 teams, pad with byes (shouldn't happen with 12 groups)
    while len(r32) < 32:
        r32.append(r32[0])

    # Knockout rounds
    current = r32[:32]
    while len(current) > 1:
        next_round = []
        for i in range(0, len(current), 2):
            winner = simulate_knockout_match(current[i], current[i+1],
                                              attack, defense, avg_atk, avg_def)
            next_round.append(winner)
        current = next_round

    return current[0]

print("Simulation functions defined.")

In [ ]:
N_SIMS = 10_000
win_counts = defaultdict(int)

np.random.seed(42)
for sim in range(N_SIMS):
    winner = simulate_tournament_2026(
        GROUPS_2026, attack, defense,
        AVG_ATTACK_GLOBAL, AVG_DEFENSE_GLOBAL
    )
    win_counts[winner] += 1

win_probs_2026 = pd.DataFrame([
    {"team": team, "win_probability": count / N_SIMS}
    for team, count in win_counts.items()
]).sort_values("win_probability", ascending=False).reset_index(drop=True)

print(f"=== 2026 World Cup — Tournament Winner Probabilities ({N_SIMS:,} simulations) ===")
display(win_probs_2026.head(16).assign(
    win_probability=lambda d: d["win_probability"].map("{:.1%}".format)
))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
top15 = win_probs_2026.head(15).copy()
ax.barh(top15["team"][::-1], top15["win_probability"][::-1], color="#4A90D9")
ax.set_xlabel("Probability of Winning Tournament")
ax.set_title(f"2026 World Cup — Predicted Win Probabilities ({N_SIMS:,} Monte Carlo simulations)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))

for i, (_, row) in enumerate(top15[::-1].iterrows()):
    ax.text(row["win_probability"] + 0.002, i, f"{row['win_probability']:.1%}",
            va="center", fontsize=9)

plt.tight_layout()
plt.show()

## 8. LightGBM Ranker — Award Predictions

**Why LightGBM Ranker instead of Ridge regression:**
Award prediction is a ranking problem (who ranked 1st?), not a regression problem
(predict a continuous score). LightGBM Ranker optimises pairwise ranking loss (LambdaRank),
which directly targets the right objective.

**Known limitation:** ~22 labelled award winners per category is a small training set.
Don't over-interpret feature importances. A simple goals-only baseline is included
for comparison — if the ranker doesn't beat it, use the baseline.

**Young Player eligibility:** U23 at tournament start (strict — not U21 — to match FIFA rules
which have varied between U21 and U23 across tournaments).

In [ ]:
!pip install lightgbm --quiet

import lightgbm as lgb

FEAT_COLS_PLAYER = [
    "goals_scored", "penalty_goals", "open_play_goals",
    "matches_played", "matches_started", "wc_exp", "age"
]

def prepare_player_data(df, feature_cols):
    df = df.copy()
    for col in feature_cols:
        if col not in df.columns:
            df[col] = 0
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    return df

player_features = prepare_player_data(player_features, FEAT_COLS_PLAYER)

# Train/test split for award models: train on pre-2018, test on 2018+2022
AWARD_TEST_YEARS  = [2018, 2022]
AWARD_TRAIN_YEARS = [y for y in player_features["year"].unique() if y not in AWARD_TEST_YEARS]

train_pl = player_features[player_features["year"].isin(AWARD_TRAIN_YEARS)].copy()
test_pl  = player_features[player_features["year"].isin(AWARD_TEST_YEARS)].copy()

print(f"Train players : {len(train_pl):,}  (tournaments: {sorted(AWARD_TRAIN_YEARS)[-5:]}...)")
print(f"Test players  : {len(test_pl):,}   (tournaments: {AWARD_TEST_YEARS})")
print(f"Features      : {FEAT_COLS_PLAYER}")

In [ ]:
def train_lgbm_ranker(train_df, feature_cols, target_col,
                      age_filter=None, n_estimators=200):
    """
    Train LightGBM Ranker on historical award data.
    Groups = one per tournament-year so the model learns relative ranking within each WC.
    age_filter: if set (e.g. 23), only include players <= that age.
    """
    df = train_df.copy()
    if age_filter is not None and "age" in df.columns:
        df = df[df["age"] <= age_filter]
    df = df.dropna(subset=feature_cols + [target_col])

    # Sort by year so group sizes are contiguous
    df = df.sort_values("year").reset_index(drop=True)
    group_sizes = df.groupby("year").size().values
    X = df[feature_cols].values
    y = df[target_col].astype(int).values

    model = lgb.LGBMRanker(
        objective="lambdarank",
        n_estimators=n_estimators,
        learning_rate=0.05,
        num_leaves=15,
        min_child_samples=5,   # small given limited data
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1,
    )
    model.fit(X, y, group=group_sizes)
    return model


def predict_award_ranker(model, test_df, feature_cols, target_col,
                          age_filter=None):
    """Score all players per tournament; rank by score; report top picks vs actual."""
    df = test_df.copy()
    if age_filter is not None and "age" in df.columns:
        df = df[df["age"] <= age_filter]
    df = df.dropna(subset=feature_cols)

    X = df[feature_cols].values
    df["ranker_score"] = model.predict(X)

    results = []
    for yr in sorted(df["year"].unique()):
        yr_df = df[df["year"] == yr].sort_values("ranker_score", ascending=False).reset_index(drop=True)
        yr_df["rank"] = yr_df.index + 1

        winners = yr_df[yr_df[target_col] == True]
        if len(winners) > 0:
            actual_rank = int(winners.iloc[0]["rank"])
            actual_name = winners.iloc[0].get("player_name","Unknown")
        else:
            actual_rank = None
            actual_name = "Unknown"

        results.append({
            "year":         yr,
            "model_pick":   yr_df.iloc[0].get("player_name","?"),
            "model_team":   yr_df.iloc[0].get("team_name","?"),
            "actual_winner":actual_name,
            "actual_rank":  actual_rank,
            "correct":      actual_rank == 1,
        })
        print(f"  {yr} | Model: {yr_df.iloc[0].get('player_name','?'):<25}  Actual: {actual_name}  (ranked #{actual_rank})")

    return pd.DataFrame(results)


def goals_baseline(test_df, target_col, age_filter=None):
    """Simplest possible baseline: pick player with most goals."""
    df = test_df.copy()
    if age_filter is not None and "age" in df.columns:
        df = df[df["age"] <= age_filter]
    results = []
    for yr in sorted(df["year"].unique()):
        yr_df = df[df["year"]==yr].sort_values("goals_scored", ascending=False).reset_index(drop=True)
        winners = yr_df[yr_df[target_col] == True]
        actual_rank = int(winners.iloc[0]["rank"]) if len(winners) > 0 and "rank" in yr_df.columns else None
        yr_df["rank"] = yr_df.index + 1
        winners = yr_df[yr_df[target_col] == True]
        actual_rank = int(winners.iloc[0]["rank"]) if len(winners) > 0 else None
        results.append({
            "year": yr,
            "correct": actual_rank == 1
        })
    return pd.DataFrame(results)

print("LightGBM Ranker functions defined.")

In [ ]:
# ── Golden Boot ─────────────────────────────────────────────────────────────
print("=== Golden Boot — LightGBM Ranker ===")
gb_model   = train_lgbm_ranker(train_pl, FEAT_COLS_PLAYER, "won_golden_boot")
gb_results = predict_award_ranker(gb_model, test_pl, FEAT_COLS_PLAYER, "won_golden_boot")

gb_baseline = goals_baseline(test_pl, "won_golden_boot")
print(f"\nRanker accuracy  : {gb_results['correct'].mean():.0%}  ({gb_results['correct'].sum()}/{len(gb_results)})")
print(f"Goals baseline   : {gb_baseline['correct'].mean():.0%}")
print("→ Note: shared Golden Boots (tied goals) counted as 1 winner in dataset")

# ── Best Player ──────────────────────────────────────────────────────────────
print("\n=== Best Player (Golden Ball) — LightGBM Ranker ===")
bp_model   = train_lgbm_ranker(train_pl, FEAT_COLS_PLAYER, "won_best_player")
bp_results = predict_award_ranker(bp_model, test_pl, FEAT_COLS_PLAYER, "won_best_player")
bp_baseline = goals_baseline(test_pl, "won_best_player")
print(f"\nRanker accuracy  : {bp_results['correct'].mean():.0%}")
print(f"Goals baseline   : {bp_baseline['correct'].mean():.0%}")
print("→ Best Player is jury-voted; stats explain only part of the outcome.")

# ── Young Player ─────────────────────────────────────────────────────────────
print("\n=== Young Player (Best Young Player) — LightGBM Ranker (U23) ===")
yp_model   = train_lgbm_ranker(train_pl, FEAT_COLS_PLAYER, "won_young_player", age_filter=23)
yp_results = predict_award_ranker(yp_model, test_pl, FEAT_COLS_PLAYER, "won_young_player", age_filter=23)
yp_baseline = goals_baseline(test_pl[test_pl["age"] <= 23] if "age" in test_pl.columns else test_pl,
                              "won_young_player")
print(f"\nRanker accuracy  : {yp_results['correct'].mean():.0%}")
print(f"Goals baseline   : {yp_baseline['correct'].mean():.0%}")

## 9. 2026 Award Predictions

The award models were trained on WC-only player stats (goals, appearances).
For 2026 we don't have tournament stats yet — the ranking here is based on
**pre-tournament form**: goals in qualifying and recent international matches.
This is intentionally stated as speculative.

In [ ]:
# Retrain award models on ALL historical data (including 2018, 2022)
gb_model_full = train_lgbm_ranker(player_features[player_features["year"] < 2026],
                                   FEAT_COLS_PLAYER, "won_golden_boot")
bp_model_full = train_lgbm_ranker(player_features[player_features["year"] < 2026],
                                   FEAT_COLS_PLAYER, "won_best_player")
yp_model_full = train_lgbm_ranker(player_features[player_features["year"] < 2026],
                                   FEAT_COLS_PLAYER, "won_young_player", age_filter=23)

print("Award models retrained on full historical data.")
print("\nNote: 2026 award predictions require 2026 squad data (goals, appearances).")
print("These will be meaningful only after populating player_features with 2026 qualifying stats.")
print("To add 2026 player data: extend player_features.csv with 2026 rows from qualifying data.")

## 10. Feature Importance — Award Models

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Award Model — LightGBM Feature Importance", fontsize=13)

for ax, (model, title) in zip(axes, [
    (gb_model,   "Golden Boot"),
    (bp_model,   "Best Player"),
    (yp_model,   "Young Player"),
]):
    importance = pd.Series(model.feature_importances_, index=FEAT_COLS_PLAYER).sort_values()
    ax.barh(importance.index, importance.values, color="#4A90D9")
    ax.set_title(title)
    ax.set_xlabel("Feature Importance (gain)")

plt.tight_layout()
plt.show()
print("\nHigher = more important for ranking. Note: with ~20 training tournaments,")
print("take these importances as indicative, not definitive.")

## 11. 2026 Predictions Summary

In [ ]:
print("=" * 60)
print("  FIFA WORLD CUP 2026 — MODEL PREDICTIONS")
print("=" * 60)

print("\n🏆 Tournament Winner (Monte Carlo, 10,000 simulations)")
for _, row in win_probs_2026.head(5).iterrows():
    print(f"   {row['team']:<30} {row['win_probability']:.1%}")

print("\nModel validation (LOO cross-validation 1998–2022):")
print(f"   Mean accuracy  : {loo_df['accuracy'].mean():.1%}")
print(f"   Mean Brier     : {loo_df['brier'].mean():.3f}  (naive = 0.333)")
print(f"   Skill score    : {1 - loo_df['brier'].mean() / 0.333:.3f}")

print("\n👟 Golden Boot, ⭐ Best Player, 🌟 Young Player:")
print("   Populate 2026 player_features.csv with qualifying stats to generate these.")

print("\n" + "=" * 60)
print("HONEST CAVEATS")
print("=" * 60)
print("1. Poisson model ignores Elo/form features — team strength")
print("   is estimated from WC match history only, not full intl form.")
print("2. Award models trained on ~20 labelled examples per category.")
print("3. 2026 bracket seeding is simplified (no actual FIFA path rules).")
print("4. Group draw verified manually — confirm against official FIFA draw.")

## 12. Save Outputs

In [ ]:
from google.colab import files as colab_files

win_probs_2026.to_csv("win_probabilities_2026.csv", index=False)
loo_df.to_csv("loo_validation_results.csv", index=False)
gb_results.to_csv("golden_boot_test_results.csv", index=False)
bp_results.to_csv("best_player_test_results.csv", index=False)
yp_results.to_csv("young_player_test_results.csv", index=False)

for fname in ["win_probabilities_2026.csv","loo_validation_results.csv",
              "golden_boot_test_results.csv","best_player_test_results.csv",
              "young_player_test_results.csv"]:
    colab_files.download(fname)
    print(f"Saved: {fname}")